<a href="https://colab.research.google.com/github/sergiocostaifes/PPCOMP_DM/blob/main/notebooks/08_model_refinement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 08_model_refinement.ipynb — Robustez, Comparação de Modelos e Interpretabilidade

## Por que este Notebook 08 existe (justificativa)

Os Notebooks 01–07 consolidaram um pipeline reprodutível e produziram um baseline supervisionado (Random Forest) com split temporal adequado. O Notebook 07 revelou um achado científico central:

- **DURING** é altamente separável com as features atuais.
- **BEFORE** apresenta baixa separabilidade.

Este Notebook 08 não tem como objetivo “otimizar score”, mas **testar robustez e validade** das conclusões do Notebook 07, respondendo se os achados persistem sob:

1) **Famílias distintas de classificadores** (linear vs ensemble vs boosting)  
2) **Validação temporal mais robusta** (corte fixo + TimeSeriesSplit)  
3) **Análise de importância de atributos**, para sustentar interpretabilidade e discussão científica.

Essa estratégia é metodologicamente preferível, neste momento do projeto, a introduzir modelos sequenciais (LSTM/Transformers), pois:
- preserva escopo e reprodutibilidade do pipeline;
- fornece evidência comparativa sólida;
- mantém o foco na validade experimental.

## Perguntas de pesquisa (RQ)

**RQ1 — Robustez ao algoritmo:**  
A separabilidade observada no Notebook 07 (DURING forte e BEFORE fraco) é estrutural dos dados/definições ou específica do Random Forest?

**RQ2 — Robustez ao split temporal:**  
O desempenho é estável quando avaliamos por corte temporal fixo e por TimeSeriesSplit?

**RQ3 — Interpretabilidade:**  
Quais features mais discriminam DURING e essa importância é consistente entre modelos?

## Dados e tarefas

Entrada:
- `window_5min_labeled.parquet` (Notebook 06)

Tarefas supervisionadas:
1) **Binária:** `event = 1` se state ∈ {BEFORE, DURING, AFTER}, senão `0` (NORMAL).
2) **Multiclasse:** `state ∈ {NORMAL, BEFORE, DURING, AFTER}`.

Split temporal:
- **Corte fixo** (80/20) como referência.
- **TimeSeriesSplit** (5 folds) para robustez.

Modelos comparados (3 famílias):
- Logistic Regression (baseline linear)
- Random Forest (baseline ensemble)
- HistGradientBoosting (boosting; evita dependências externas)

Saídas:
- `08_model_refinement_summary.json`
- `rf_binary.joblib`, `rf_multiclass.joblib` (já existiam; aqui podem ser regravados/atualizados)
- `lr_binary.joblib`, `lr_multiclass.joblib`
- `hgb_binary.joblib`, `hgb_multiclass.joblib`

## Métricas

Binário:
- ROC-AUC
- Precision, Recall, F1
- Matriz de confusão

Multiclasse:
- Accuracy
- Macro-F1, Weighted-F1
- Recall por classe (ênfase em BEFORE e DURING)
- Matriz de confusão

Interpretabilidade:
- **Permutation importance** (top-10), no conjunto de teste do corte fixo
- Importância nativa quando aplicável (secundária)

Observação:
- Este notebook mantém as mesmas features do Notebook 05 e a mesma rotulagem do Notebook 06, evitando vazamento temporal e garantindo comparabilidade.

In [1]:
# ============================================================
# 08_model_refinement.ipynb
# Robustez, comparação de modelos e interpretabilidade
# (binário vs multiclasse) + split temporal + TimeSeriesSplit
# ============================================================

# -----------------------------
# 0) BOOTSTRAP (Colab + Repo)
# -----------------------------
from pathlib import Path
import os
import sys
import subprocess
import importlib
import random
import numpy as np
import pandas as pd
import json

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("[Bootstrap] Google Drive já montado.")

REPO_DIR = Path("/content/drive/MyDrive/Mestrado/PPCOMP_DM")
GITHUB_REPO = "https://github.com/sergiocostaifes/PPCOMP_DM.git"

if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    print(f"[Bootstrap] Clonando repositório em: {REPO_DIR}")
    subprocess.run(["git", "clone", GITHUB_REPO, str(REPO_DIR)], check=True)
else:
    try:
        print("[Bootstrap] Atualizando repositório (git pull).")
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
    except Exception as e:
        print("[Bootstrap] Aviso: não foi possível atualizar via git pull:", e)

os.chdir(str(REPO_DIR))
print("[Bootstrap] CWD =", os.getcwd())

repo_str = str(REPO_DIR)
if repo_str not in sys.path:
    sys.path.insert(0, repo_str)

importlib.invalidate_caches()

from src.paths import FEATURES_PATH, REPORTS_PATH, MODELS_PATH, ensure_dirs
ensure_dirs()

print("FEATURES_PATH =", FEATURES_PATH)
print("REPORTS_PATH  =", REPORTS_PATH)
print("MODELS_PATH   =", MODELS_PATH)

def log(msg: str) -> None:
    print(f"[08_model_refinement] {msg}")

# -----------------------------
# 1) Imports ML
# -----------------------------
import joblib

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

from sklearn.inspection import permutation_importance

# -----------------------------
# 2) Carregar dataset rotulado
# -----------------------------
DATA_FILE = FEATURES_PATH / "window_5min_labeled.parquet"
assert DATA_FILE.exists(), f"Arquivo não encontrado: {DATA_FILE}"

df = pd.read_parquet(DATA_FILE).sort_values("bucket_id").reset_index(drop=True)
log(f"Dataset rotulado: shape={df.shape}")

assert "state" in df.columns, "Coluna 'state' ausente."
assert "bucket_id" in df.columns, "Coluna 'bucket_id' ausente."

# -----------------------------
# 3) Preparar features e targets
# -----------------------------
# Remover colunas de label / não numéricas
drop_cols = {"state", "bucket_id"}
if "is_critical" in df.columns:
    drop_cols.add("is_critical")

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
X_cols = [c for c in numeric_cols if c not in drop_cols]
if not X_cols:
    raise ValueError("Nenhuma feature numérica encontrada após filtragem.")

X = df[X_cols].copy()
y_multi = df["state"].astype(str).copy()
y_bin = (y_multi != "NORMAL").astype(int)

labels = ["NORMAL", "BEFORE", "DURING", "AFTER"]

log(f"Features: {len(X_cols)}")
log(f"Classes: {sorted(y_multi.unique().tolist())}")
log(f"Distribuição total: {df['state'].value_counts().to_dict()}")

# -----------------------------
# 4) Split temporal (corte fixo 80/20)
# -----------------------------
TEST_RATIO = 0.20
n = len(df)
test_size = int(np.ceil(n * TEST_RATIO))
train_end = n - test_size

X_train, X_test = X.iloc[:train_end], X.iloc[train_end:]
y_train_multi, y_test_multi = y_multi.iloc[:train_end], y_multi.iloc[train_end:]
y_train_bin, y_test_bin = y_bin.iloc[:train_end], y_bin.iloc[train_end:]

log(f"Split fixed: train={len(X_train)} test={len(X_test)}")

# -----------------------------
# 5) Definição de modelos (3 famílias)
# -----------------------------
# Logistic Regression: usa scaler
lr_bin = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        random_state=SEED,
        max_iter=5000,
        class_weight="balanced"
    ))
])

lr_multi = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        random_state=SEED,
        max_iter=5000,
        class_weight="balanced",
        multi_class="auto"
    ))
])

rf_bin = RandomForestClassifier(
    n_estimators=400,
    random_state=SEED,
    n_jobs=-1,
    class_weight="balanced_subsample"
)

rf_multi = RandomForestClassifier(
    n_estimators=500,
    random_state=SEED,
    n_jobs=-1,
    class_weight="balanced_subsample"
)

hgb_bin = HistGradientBoostingClassifier(
    random_state=SEED,
    max_depth=None,
    learning_rate=0.1
)

hgb_multi = HistGradientBoostingClassifier(
    random_state=SEED,
    max_depth=None,
    learning_rate=0.1
)

models = {
    "logreg": {"bin": lr_bin, "multi": lr_multi},
    "rf": {"bin": rf_bin, "multi": rf_multi},
    "hgb": {"bin": hgb_bin, "multi": hgb_multi},
}

# -----------------------------
# 6) Funções de avaliação
# -----------------------------
def eval_binary(model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)

    proba = None
    if hasattr(model, "predict_proba"):
        try:
            proba = model.predict_proba(X_te)[:, 1]
        except Exception:
            proba = None

    acc = accuracy_score(y_te, pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_te, pred, average="binary", zero_division=0)
    cm = confusion_matrix(y_te, pred).tolist()

    auc = None
    if proba is not None and len(np.unique(y_te)) == 2:
        try:
            auc = float(roc_auc_score(y_te, proba))
        except Exception:
            auc = None

    return {
        "acc": float(acc),
        "precision": float(prec),
        "recall": float(rec),
        "f1": float(f1),
        "roc_auc": auc,
        "confusion_matrix": cm
    }

def eval_multiclass(model, X_tr, y_tr, X_te, y_te, labels_order):
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)

    acc = accuracy_score(y_te, pred)

    prec_m, rec_m, f1_m, _ = precision_recall_fscore_support(
        y_te, pred, average="macro", zero_division=0, labels=labels_order
    )
    prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(
        y_te, pred, average="weighted", zero_division=0, labels=labels_order
    )

    # recall por classe
    prec_c, rec_c, f1_c, sup_c = precision_recall_fscore_support(
        y_te, pred, average=None, zero_division=0, labels=labels_order
    )
    per_class = {
        lbl: {
            "precision": float(p),
            "recall": float(r),
            "f1": float(f),
            "support": int(s)
        }
        for lbl, p, r, f, s in zip(labels_order, prec_c, rec_c, f1_c, sup_c)
    }

    cm = confusion_matrix(y_te, pred, labels=labels_order).tolist()
    report_txt = classification_report(y_te, pred, labels=labels_order, zero_division=0)

    return {
        "acc": float(acc),
        "macro_f1": float(f1_m),
        "weighted_f1": float(f1_w),
        "confusion_matrix": cm,
        "labels_order": labels_order,
        "per_class": per_class,
        "classification_report_text": report_txt
    }

def tscv_eval_multiclass(model_builder, X_all, y_all, n_splits=5):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    folds = []
    for fold, (tr, te) in enumerate(tscv.split(X_all), start=1):
        model = model_builder()
        X_tr, X_te = X_all.iloc[tr], X_all.iloc[te]
        y_tr, y_te = y_all.iloc[tr], y_all.iloc[te]
        model.fit(X_tr, y_tr)
        pred = model.predict(X_te)
        macro_f1 = precision_recall_fscore_support(y_te, pred, average="macro", zero_division=0)[2]
        folds.append({"fold": fold, "macro_f1": float(macro_f1), "n_train": int(len(tr)), "n_test": int(len(te))})
    return folds

# -----------------------------
# 7) Rodar avaliações (corte fixo)
# -----------------------------
results = {
    "meta": {
        "seed": SEED,
        "rows_total": int(n),
        "rows_train": int(len(X_train)),
        "rows_test": int(len(X_test)),
        "test_ratio": TEST_RATIO,
        "n_features": int(len(X_cols)),
        "features": X_cols,
        "class_distribution_total": df["state"].value_counts().to_dict()
    },
    "fixed_split": {"binary": {}, "multiclass": {}},
    "tscv_multiclass": {}
}

for name, pack in models.items():
    log(f"== Modelo: {name} (BIN) ==")
    r_bin = eval_binary(pack["bin"], X_train, y_train_bin, X_test, y_test_bin)
    results["fixed_split"]["binary"][name] = r_bin
    log(r_bin)

    log(f"== Modelo: {name} (MULTI) ==")
    r_multi = eval_multiclass(pack["multi"], X_train, y_train_multi, X_test, y_test_multi, labels)
    results["fixed_split"]["multiclass"][name] = r_multi
    log({k: r_multi[k] for k in ["acc", "macro_f1", "weighted_f1"]})
    # imprimir relatório multiclass de um modelo por vez (opcional)
    print(f"\n=== {name.upper()} MULTICLASS REPORT (fixed split) ===\n")
    print(r_multi["classification_report_text"])

# -----------------------------
# 8) TimeSeriesSplit (apenas multiclasse) — robustez
# -----------------------------
def build_lr_multi():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            random_state=SEED,
            max_iter=5000,
            class_weight="balanced",
            multi_class="auto"
        ))
    ])

def build_rf_multi():
    return RandomForestClassifier(
        n_estimators=400,
        random_state=SEED,
        n_jobs=-1,
        class_weight="balanced_subsample"
    )

def build_hgb_multi():
    return HistGradientBoostingClassifier(
        random_state=SEED,
        learning_rate=0.1
    )

results["tscv_multiclass"]["logreg"] = tscv_eval_multiclass(build_lr_multi, X, y_multi, n_splits=5)
results["tscv_multiclass"]["rf"]     = tscv_eval_multiclass(build_rf_multi, X, y_multi, n_splits=5)
results["tscv_multiclass"]["hgb"]    = tscv_eval_multiclass(build_hgb_multi, X, y_multi, n_splits=5)

# -----------------------------
# 9) Permutation importance (fixed split) — interpretabilidade
# -----------------------------
# Foco: modelo RF multiclasse (robusto e já usado) como referência interpretável
log("Calculando permutation importance (RF multiclass) no conjunto de teste...")

rf_ref = models["rf"]["multi"]
rf_ref.fit(X_train, y_train_multi)

perm = permutation_importance(
    rf_ref,
    X_test,
    y_test_multi,
    n_repeats=10,
    random_state=SEED,
    n_jobs=-1
)

pi = pd.DataFrame({
    "feature": X_cols,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False).reset_index(drop=True)

top10 = pi.head(10).copy()
results["permutation_importance_top10"] = top10.to_dict(orient="records")

print("\n=== TOP-10 Permutation Importance (RF multiclass) ===\n")
print(top10)

# -----------------------------
# 10) Salvar modelos (joblib)
# -----------------------------
# Salvar os modelos treinados no fixed split (binário e multiclass)
# Observação: para pipelines (logreg), salvamos o objeto inteiro.

out_paths = {}
for name, pack in models.items():
    # binário
    pack["bin"].fit(X_train, y_train_bin)
    p_bin = MODELS_PATH / f"{name}_binary.joblib"
    joblib.dump(pack["bin"], p_bin)

    # multiclasse
    pack["multi"].fit(X_train, y_train_multi)
    p_multi = MODELS_PATH / f"{name}_multiclass.joblib"
    joblib.dump(pack["multi"], p_multi)

    out_paths[name] = {"binary": str(p_bin), "multiclass": str(p_multi)}

results["models"] = out_paths

# -----------------------------
# 11) Persistir summary JSON
# -----------------------------
summary_file = REPORTS_PATH / "08_model_refinement_summary.json"
summary_file.write_text(json.dumps(results, indent=2, ensure_ascii=False))

log("Notebook 08 finalizado com sucesso.")
log(f"Summary: {summary_file}")

# Exibir um resumo compacto das métricas principais (fixed split)
print("\n=== RESUMO (fixed split) — BINÁRIO ===")
for name in models.keys():
    r = results["fixed_split"]["binary"][name]
    print(name, {"f1": round(r["f1"], 4), "recall": round(r["recall"], 4), "auc": None if r["roc_auc"] is None else round(r["roc_auc"], 4)})

print("\n=== RESUMO (fixed split) — MULTICLASS ===")
for name in models.keys():
    r = results["fixed_split"]["multiclass"][name]
    print(name, {"macro_f1": round(r["macro_f1"], 4), "weighted_f1": round(r["weighted_f1"], 4)})

# Mostrar TSCV macro_f1 por fold
print("\n=== TimeSeriesSplit (5 folds) — macro_f1 por fold (multiclasse) ===")
for name, folds in results["tscv_multiclass"].items():
    vals = [f["macro_f1"] for f in folds]
    print(name, {"mean": float(np.mean(vals)), "std": float(np.std(vals)), "folds": vals})

Mounted at /content/drive
[Bootstrap] Atualizando repositório (git pull).
[Bootstrap] CWD = /content/drive/MyDrive/Mestrado/PPCOMP_DM
FEATURES_PATH = /content/drive/MyDrive/Mestrado/02-datasets/03-features
REPORTS_PATH  = /content/drive/MyDrive/Mestrado/04-reports
MODELS_PATH   = /content/drive/MyDrive/Mestrado/03-models
[08_model_refinement] Dataset rotulado: shape=(8914, 27)
[08_model_refinement] Features: 24
[08_model_refinement] Classes: ['AFTER', 'BEFORE', 'DURING', 'NORMAL']
[08_model_refinement] Distribuição total: {'NORMAL': 7201, 'BEFORE': 854, 'AFTER': 659, 'DURING': 200}
[08_model_refinement] Split fixed: train=7131 test=1783
[08_model_refinement] == Modelo: logreg (BIN) ==
[08_model_refinement] {'acc': 0.7324733595064498, 'precision': 0.5012437810945274, 'recall': 0.8413361169102297, 'f1': 0.6282151208106002, 'roc_auc': 0.8743099760492847, 'confusion_matrix': [[903, 401], [76, 403]]}
[08_model_refinement] == Modelo: logreg (MULTI) ==


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


[08_model_refinement] {'acc': 0.6085249579360628, 'macro_f1': 0.6184907667332517, 'weighted_f1': 0.6580374375301657}

=== LOGREG MULTICLASS REPORT (fixed split) ===

              precision    recall  f1-score   support

      NORMAL       0.94      0.59      0.73      1304
      BEFORE       0.22      0.60      0.32       242
      DURING       0.81      1.00      0.90        61
       AFTER       0.47      0.61      0.53       176

    accuracy                           0.61      1783
   macro avg       0.61      0.70      0.62      1783
weighted avg       0.79      0.61      0.66      1783

[08_model_refinement] == Modelo: rf (BIN) ==
[08_model_refinement] {'acc': 0.8625911385305665, 'precision': 0.95703125, 'recall': 0.511482254697286, 'f1': 0.6666666666666666, 'roc_auc': 0.878165144664882, 'confusion_matrix': [[1293, 11], [234, 245]]}
[08_model_refinement] == Modelo: rf (MULTI) ==
[08_model_refinement] {'acc': 0.8295008412787437, 'macro_f1': 0.657889314870657, 'weighted_f1': 0.774

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and wi

[08_model_refinement] Calculando permutation importance (RF multiclass) no conjunto de teste...

=== TOP-10 Permutation Importance (RF multiclass) ===

            feature  importance_mean  importance_std
0    rolling_std_1h         0.116601        0.005246
1   rolling_mean_1h         0.003197        0.002299
2  event_LOST_count         0.001066        0.000586
3        n_machines         0.000673        0.000654
4  event_FAIL_count         0.000617        0.000685
5     n_collections         0.000561        0.000502
6             lag_3         0.000505        0.000885
7     zscore_global         0.000505        0.000685
8        pct_change         0.000112        0.000489
9          n_failed         0.000112        0.000744


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


[08_model_refinement] Notebook 08 finalizado com sucesso.
[08_model_refinement] Summary: /content/drive/MyDrive/Mestrado/04-reports/08_model_refinement_summary.json

=== RESUMO (fixed split) — BINÁRIO ===
logreg {'f1': 0.6282, 'recall': 0.8413, 'auc': 0.8743}
rf {'f1': 0.6667, 'recall': 0.5115, 'auc': 0.8782}
hgb {'f1': 0.6951, 'recall': 0.5449, 'auc': 0.8354}

=== RESUMO (fixed split) — MULTICLASS ===
logreg {'macro_f1': 0.6185, 'weighted_f1': 0.658}
rf {'macro_f1': 0.6579, 'weighted_f1': 0.7742}
hgb {'macro_f1': 0.6582, 'weighted_f1': 0.7804}

=== TimeSeriesSplit (5 folds) — macro_f1 por fold (multiclasse) ===
logreg {'mean': 0.5781660461844348, 'std': 0.07800333412858959, 'folds': [0.6341016035451597, 0.42909902403783895, 0.6199954250710574, 0.5721375368950694, 0.6354966413730486]}
rf {'mean': 0.6557455650725952, 'std': 0.016747213476797848, 'folds': [0.6315465760986967, 0.643180850416692, 0.6784207488669461, 0.6675582376590987, 0.6580214123215424]}
hgb {'mean': 0.6780771518086763, 

## Achados do Notebook 08 — Robustez, Comparação de Modelos e Interpretabilidade

### Objetivo

Avaliar se os achados do Notebook 07 são robustos:

1. Entre diferentes famílias de classificadores.
2. Sob validação temporal mais rigorosa.
3. Sob análise de importância de atributos.

---

## 1. Robustez ao Algoritmo

### 1.1 Tarefa Binária (evento vs NORMAL)

F1-score (corte fixo):

- Logistic Regression: 0.628
- Random Forest: 0.667
- HistGradientBoosting: 0.695

Conclusão:

A separabilidade entre regime normal e regime de evento
é estrutural dos dados, não dependente de um único algoritmo.

Modelos baseados em árvore apresentam melhor equilíbrio
entre precisão e recall, enquanto o modelo linear apresenta
alto recall mas menor precisão.

---

### 1.2 Tarefa Multiclasse

Macro-F1 (corte fixo):

- Logistic Regression: 0.618
- Random Forest: 0.658
- HistGradientBoosting: 0.658

TimeSeriesSplit (macro-F1 médio):

- Logistic Regression: ~0.578 (maior variabilidade)
- Random Forest: ~0.656 (estável)
- HistGradientBoosting: ~0.678 (melhor desempenho médio)

Conclusão:

O boosting apresentou maior estabilidade temporal.
Os resultados confirmam que:

- DURING é altamente separável.
- BEFORE permanece estruturalmente difícil.

---

## 2. Análise por Classe

### DURING

- Recall = 1.0 em todos os modelos.
- F1 ≈ 1.0 (árvores).

Conclusão:

A fase crítica possui assinatura estatística forte e consistente.

---

### BEFORE

- Logistic Regression: recall ≈ 0.60, baixa precisão.
- Random Forest: recall ≈ 0.02.
- HistGradientBoosting: recall ≈ 0.00.

Conclusão:

A fase BEFORE não é bem modelada com as features atuais.
Isso sugere:

- ausência de sinal estatístico claro pré-evento;
- necessidade de features temporais mais longas;
- ou adoção futura de modelos sequenciais.

Este é um achado científico relevante.

---

## 3. Interpretabilidade

Permutation Importance (RF multiclass):

Feature dominante:

- rolling_std_1h (importância ≈ 0.1166)

Demais variáveis apresentam impacto significativamente menor.

Conclusão:

A variância local em janela de 1 hora é o principal
discriminador de regime.

Isso valida empiricamente a definição estatística baseada
em desvio-padrão (μ + 2σ), fortalecendo a coerência metodológica.

---

## 4. Robustez Temporal

A avaliação com TimeSeriesSplit confirma:

- estabilidade do desempenho das árvores;
- maior variabilidade do modelo linear;
- manutenção do padrão DURING forte / BEFORE fraco.

Não há indícios de overfitting grave no corte fixo.

---

## 5. Conclusões do Notebook 08

1. A separabilidade de DURING é estrutural.
2. A dificuldade em detectar BEFORE é robusta entre modelos.
3. O boosting apresenta melhor estabilidade temporal.
4. rolling_std_1h é a variável mais discriminante.
5. O pipeline mantém coerência estatística entre definição
   de episódio e modelagem supervisionada.

O Notebook 08 fortalece a validade experimental do projeto
sem aumentar complexidade desnecessária.